In [16]:
#Import Necessary Packages/Modules
from skeLCS import eLCS # Run "pip install scikit-eLCS" to install necessary module
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score

In [17]:
# Read file into a table. Turns off default behavior that reads "none" as missing
df = pd.read_csv("medical_insurance.csv", keep_default_na=False, na_values=[""])

In [18]:
# eLCS can only predict classes, so "risk_score" will be cut into 3 groups
df["risk_band"] = pd.qcut(df["risk_score"], 3, labels=False) # labels=False gives plain integers, which are better for machine learning
y = df["risk_band"].values # converts the column to a NumPy array

In [19]:
# Define what the leaky columns are
LEAKAGE_COLS = ["is_high_risk", "annual_premium", "monthly_premium", "annual_medical_cost", "claims_count", "avg_claim_amount", "total_claims_paid",]

# Keeps every column apart from the ones that come as a result of "risk_score" 
X = df.drop(columns=LEAKAGE_COLS + ["risk_score", "risk_band", "person_id"])

In [20]:
# eLCS needs numbers, so every text column becomes integer codes
for col in X.select_dtypes(exclude="number"):
    X[col] = X[col].astype("category").cat.codes

In [ ]:
feature_names = X.columns   # Get the column names of the features
tree.fit(X, y)  # Train a decision tree on the full dataset
print(pd.Series(tree.feature_importances_, index=feature_names).sort_values(ascending=False).head(10))  # feature_importances_ gives one number per feature that add up to 1

chronic_count     0.404078
age               0.366792
smoker            0.160757
bmi               0.068372
sex               0.000000
income            0.000000
urban_rural       0.000000
education         0.000000
marital_status    0.000000
household_size    0.000000
dtype: float64


In [22]:
# Converts the table to a plain array of decimals
X = X.values.astype(float)

In [11]:
# Run and score model
model = eLCS()
cv = StratifiedKFold(3, shuffle=True, random_state=42)
print(cross_val_score(model, X, y, cv=cv).mean())

0.4818330481588425


In [13]:
# Comparison with decision tree
tree = DecisionTreeClassifier(max_depth=5)
print(cross_val_score(tree, X, y, cv=cv, scoring="balanced_accuracy").mean())

0.9528903015501881


In [14]:
model = eLCS()
model.fit(X, y)
print(model.get_final_accuracy(), model.get_final_instance_coverage())

0.5760408766661856 0.63437


In [23]:
SELECTED = ["chronic_count", "age", "smoker", "bmi"]

X = df[SELECTED].copy()
for col in X.select_dtypes(exclude="number"):
    X[col] = X[col].astype("category").cat.codes
X = X.values.astype(float)

model = eLCS()
cv = StratifiedKFold(3, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv)
print(scores, scores.mean())

[0.83002358 0.81894164 0.80932007] 0.8194284321406031
